In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics', 'supervision', 'scikit-learn', 'pandas',
    'opencv-python-headless'], check=True)
print("Dependencies installed")

In [ ]:
import os, sys

PROJECT_PATH = '/content/drive/MyDrive/foostats_ai'

if not os.path.exists(PROJECT_PATH):
    raise FileNotFoundError(f"Project not found at {PROJECT_PATH}. Upload foostats_ai folder to Drive.")

os.chdir(PROJECT_PATH)
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

print(f"Working dir: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
required = [
    'main.py',
    'models/best.pt',
    'input_videos/08fd33_4.mp4',
    'trackers/tracker.py',
    'teams_assigner/team_assigner.py',
    'camara_movement_estimator/camara_movement_estimator.py',
    'player_ball_assigner/player_ball_assigner.py',
    'speed_and_distance_estimator/speed_and_distance_estimator.py',
    'view_transformer/view_transformer.py',
    'utils/video_utils.py',
]

all_ok = True
for f in required:
    ok = os.path.exists(f)
    print(f"{'OK' if ok else 'MISSING'}  {f}")
    if not ok:
        all_ok = False

print("\nAll files present — ready to run." if all_ok else "\nFix missing files before running cell 6.")

In [ ]:
import time, importlib
import main as m

os.makedirs('output_videos', exist_ok=True)
os.makedirs('stubs', exist_ok=True)

start = time.time()
importlib.reload(m)
m.main()
elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)")

In [ ]:
output = 'output_videos/output_video.avi'
if os.path.exists(output):
    print(f"Output: {output} ({os.path.getsize(output)/1e6:.1f} MB)")
    print(f"Saved to Drive: {PROJECT_PATH}/output_videos/output_video.avi")
else:
    print("No output generated — check errors in cell 6.")

In [ ]:
import cv2
cap = cv2.VideoCapture('input_videos/08fd33_4.mp4')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

ms_per_frame = 20  # T4 GPU estimate
est_min = (int(90 * 60 * fps) * ms_per_frame / 1000) / 60
print(f"Clip: {total} frames ({total/fps/60:.1f} min @ {fps:.0f}fps)")
print(f"Estimated for full 90-min match: ~{est_min:.0f} min on T4")